In [ ]:
import os
os.environ ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
!nvidia-smi

In [ ]:
# !pip install chromadb

In [ ]:
# pip install unstructured

In [ ]:
# pip install InstructorEmbedding

In [ ]:
# pip install langchain
!pip install tokenizers==0.13.3

In [ ]:
import os
os.environ['HUGGINGFACEHUB_API_TOKEN']=''
os.environ['OPENAI_API_KEY'] =''

In [ ]:
from langchain.embeddings import HuggingFaceInstructEmbeddings
from langchain.document_loaders import DirectoryLoader
from langchain.embeddings import HuggingFaceHubEmbeddings
from langchain.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains.question_answering import load_qa_chain
from langchain.chains import VectorDBQA
from langchain.llms import OpenAI,HuggingFaceHub
from InstructorEmbedding import INSTRUCTOR
from transformers import pipeline

### Creating Chunks

In [ ]:
loader=DirectoryLoader('/data/hindi/thesis/Legal_QA/Legal Doc Preprocessed')
docs=loader.load()
character_text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=250)
doc_texts=character_text_splitter.split_documents(docs)

## Creating Embeddings and saving in a database

In [ ]:
vectordb = Chroma.from_documents(documents=doc_texts, embedding=embeddings, persist_directory=persist_directory)

In [ ]:
vectordb.persist()
vectordb=None

## Loading existing OpenSource Embedding

In [ ]:
embeddings = HuggingFaceInstructEmbeddings(
    query_instruction="Represent these legal documents for retrieval: ",
    model_name='hkunlp/instructor-large'
)
persist_directory = ''
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

## Loading existing OpenAI Embedding

In [ ]:
# pip install tiktoken

In [ ]:
# pip install openai

In [ ]:
from langchain.embeddings.openai import OpenAIEmbeddings

In [ ]:
persist_directory = ''
EMBEDDING_MODEL='text-embedding-ada-002'
embeddings=OpenAIEmbeddings(openai_api_key=os.environ['OPENAI_API_KEY'])
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embeddings)

In [ ]:
vectordb.as_retriever()

## OpenSource Model Trial-Flan T5

In [ ]:
from transformers import pipeline
from langchain.llms.base import LLM
import torch


In [ ]:
class customLLM(LLM):
    model_name = "google/flan-t5-base"
    pipeline = pipeline("text2text-generation", model=model_name, model_kwargs={"torch_dtype":torch.bfloat16})

    def _call(self, prompt, stop=None):
        return self.pipeline(prompt, max_length=9999)[0]["generated_text"]
 
    def _identifying_params(self):
        return {"name_of_model": self.model_name}

    def _llm_type(self):
        return "custom"


In [ ]:
llm=customLLM()
chain=load_qa_chain(llm,chain_type='stuff')

In [ ]:
prompt='''Your task is to answer a question as a legal assistant to the best of your abilities, using the context given in the document. If the country is not mentioned in the question, your response should be related to India. You have knowledge of all laws and legal judgements of India. Be detailed in your answer, provide relevant sections and caselaws in your answer only if you are confident that they are correct.
Note that if you do not know the answer, it is acceptable to say "Sorry, I don't know."
{context}
{Question:}
'''

## Flan Ul2 Trial


In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer
import torch


In [ ]:
model = T5ForConditionalGeneration.from_pretrained("google/flan-ul2", device_map="auto", load_in_8bit=True)                                                                 
tokenizer = AutoTokenizer.from_pretrained("google/flan-ul2")


In [ ]:
prompt='''
Answer the following question using the context by resoning step by step.If you don't know the answer,just say "Sorry,I dont know":\n\n
Question:{}\n\n
Context:{}
'''

In [ ]:
query='what is punishment for murder under 18 age?'
context='Punishment for murder is death'

In [ ]:
input_string=prompt.format(query,context)

In [ ]:
inputs = tokenizer(input_string, return_tensors="pt").input_ids.to("cuda")
outputs = model.generate(inputs,max_length=1000)

result=tokenizer.decode(outputs[0])
result=result.lstrip('<pad>').rstrip('</s>').strip()

In [ ]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm

In [ ]:
ground_truth=pd.read_csv('')
ground_truth.head(3)

In [ ]:
rows=[]
for i in tqdm(range(len(ground_truth))):
    query=ground_truth['question'][i].strip()
    context=ground_truth['context'][i].strip()
    input_string=prompt.format(query,context)
    inputs = tokenizer(input_string, return_tensors="pt").input_ids.to("cuda")
    outputs = model.generate(inputs,max_length=1000)
    result=tokenizer.decode(outputs[0])
    result=result.lstrip('<pad>').rstrip('</s>').strip()
    rows.append([ground_truth['title'][i],ground_truth['question'][i],ground_truth['ground_truth'][i],result,0])
    time.sleep(2)

In [ ]:
import csv
fields=['title','question','ground_truth','answer_generated','score']
with open('', 'w+') as f:
    # using csv.writer method from CSV package
    write = csv.writer(f)
    write.writerow(fields)
    write.writerows(rows)
f.close()

In [ ]:
from  transformers  import  AutoTokenizer, AutoModelWithLMHead, pipeline

In [ ]:
llm=HuggingFaceHub(repo_id='EleutherAI/gpt-j-6B')

In [ ]:
chain=load_qa_chain(llm,chain_type='stuff')

In [ ]:
prompt='''Your task is to answer a question as a legal assistant to the best of your abilities, using the context given in the document. If the country is not mentioned in the question, your response should be related to India. You have knowledge of all laws and legal judgements of India. Be detailed in your answer, provide relevant sections and caselaws in your answer only if you are confident that they are correct.
Note that if you do not know the answer, it is acceptable to say "Sorry, I don't know."
{context}
{Question:}
'''

In [ ]:
query='what is this?'
context_docs=vectordb.similarity_search(query)
result=chain.run(input_documents=context_docs,question=query)
print(result)

In [ ]:
# pip install openai

In [ ]:
llm=OpenAI(temperature=0,openai_api_key=os.environ['OPENAI_API_KEY'],model_name='text-davinci-003')
chain=load_qa_chain(llm,chain_type='stuff')

In [ ]:
prompt='''Your task is to answer a question as a legal assistant to the best of your abilities, using the context given in the document. If the country is not mentioned in the question, your response should be related to India. You have knowledge of all laws and legal judgements of India. Be detailed in your answer, provide relevant sections and caselaws in your answer only if you are confident that they are correct.
Note that if you do not know the answer, it is acceptable to say "Sorry, I don't know."
{context}
{Question:}
'''

In [ ]:
query='What is the procedure that the police must follow when they have been informed of a cognizable offence? Please provide relevant sections to bolster the answer.'
context_docs=vectordb.similarity_search(query)
result=chain.run(input_documents=context_docs,question=prompt+query)
print(result)

## LONGFORMER-BASE-4096 fine-tuned on SQuAD v1 trial

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

In [ ]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("valhalla/longformer-base-4096-finetuned-squadv1")
model = AutoModelForQuestionAnswering.from_pretrained("valhalla/longformer-base-4096-finetuned-squadv1")

In [ ]:
ground_truth=pd.read_csv('')
ground_truth.head(3)

In [ ]:
rows=[]
for i in tqdm(range(len(ground_truth))):
    query=ground_truth['question'][i].strip()
    context=ground_truth['context'][i].strip()
    encoding = tokenizer(query, context, return_tensors="pt")
    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    # print(model(input_ids, attention_mask=attention_mask))
    output = model(input_ids, attention_mask=attention_mask)
    start_scores=output['start_logits']
    end_scores=output['end_logits']
    all_tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())
    answer_tokens = all_tokens[torch.argmax(start_scores) :torch.argmax(end_scores)+1]
    result = tokenizer.decode(tokenizer.convert_tokens_to_ids(answer_tokens))
    rows.append([ground_truth['title'][i],ground_truth['question'][i],ground_truth['ground_truth'][i],result,0])
    time.sleep(2)

In [ ]:
import csv
fields=['title','question','ground_truth','answer_generated','score']
with open('', 'w+') as f:
    # using csv.writer method from CSV package
    write = csv.writer(f)
    write.writerow(fields)
    write.writerows(rows)
f.close()

## OpenAI QA Trial

In [ ]:
from langchain.llms import OpenAI
from langchain.chains import LLMChain, ConstitutionalChain
from langchain.chains.constitutional_ai.models import ConstitutionalPrinciple
from langchain import PromptTemplate


In [ ]:
llm = OpenAI(model_name='text-davinci-003',openai_api_key=os.environ['OPENAI_API_KEY'],temperature=0)

In [ ]:
qa_prompt = PromptTemplate(
    template='''Your task is to answer a question as a legal assistant to the best of your abilities. If the country is not mentioned in the question, your response should be related to India. You have knowledge of all laws and legal judgements of India. Be detailed in your answer, provide relevant sections and caselaws in your answer only if you are confident that they are correct.
Note that if you do not know the answer, it is acceptable to say "Sorry, I don't know."
Question:{query}
    ''',
    input_variables=["query"],
)

In [ ]:
qa_chain = LLMChain(llm=llm, prompt=qa_prompt)


In [ ]:
constitutional_chain = ConstitutionalChain.from_llm(
    llm=llm,
    chain=qa_chain,
    constitutional_principles=[
        ConstitutionalPrinciple(
            critique_request="Tell if this answer is good.",
            revision_request="Give a better answer.",
        )
    ],
)

In [ ]:
ground_truth=pd.read_csv('')
ground_truth.head(3)

In [ ]:
rows=[]
for i in tqdm(range(len(ground_truth))):
    query=ground_truth['question'][i].strip()
    result = constitutional_chain.run(query= query)
    rows.append([ground_truth['title'][i],ground_truth['question'][i],ground_truth['ground_truth'][i],result,0])
    time.sleep(5)

In [ ]:
import csv
fields=['title','question','ground_truth','answer_generated','score']
with open('', 'w+') as f:
    # using csv.writer method from CSV package
    write = csv.writer(f)
    write.writerow(fields)
    write.writerows(rows)
f.close()

## Testing QA System over Test Dataset

In [ ]:
import pandas as pd
import numpy as np
import time
from tqdm import tqdm

In [ ]:
ground_truth=pd.read_csv('')
ground_truth=ground_truth.drop(columns=['Unnamed: 0'])
ground_truth.head(3)

In [ ]:
!nvidia-smi

In [ ]:
rows=[]
for i in tqdm(range(len(ground_truth))):
    query='Question:'+ground_truth['question'][i].strip()
    context_docs=vectordb.similarity_search(query)
    result=chain.run(input_documents=context_docs,question=prompt+query)
    rows.append([ground_truth['title'][i],ground_truth['question'][i],ground_truth['answer'][i],result.strip(),0])
    time.sleep(5)

In [ ]:
rows[5]

In [ ]:
import csv
fields=['title','question','ground_truth','answer_generated','score']
with open('', 'w+') as f:
      
    # using csv.writer method from CSV package
    write = csv.writer(f)
      
    write.writerow(fields)
    write.writerows(rows)
f.close()

## LLama 7b 32K Testing

In [ ]:
!pip install torch==1.13.1

In [ ]:
!pip install -qU transformers accelerate einops langchain xformers bitsandbytes faiss-gpu sentence_transformers